# ⚡ AlgorimGen — Agentic / Function-Calling / Execution Model
**Model:** DeepSeek-R1 / Qwen2.5 (7B / 3B / 1.5B / 0.5B)  
**Dataset:** `algorim_execution_traces.json` + `algorim_extra.json` + `algorim_training.json` (commands category)  
**Method:** SFT with function-calling + chain-of-thought format via Unsloth + TRL  
**Special:** `/execute` `/compile` `/debug` `/imagine` tool-call format with VM traces  
**Output:** LoRA → Merged HF → GGUF F16 → GGUF Q4_K_M


## ⚙️ 1 · GPU Check

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 📦 2 · Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install datasets transformers sentencepiece huggingface_hub
print("✅ Installed")


## 📁 3 · Upload & Extract Datasets

In [ ]:
import os, zipfile, json, glob
from google.colab import files

print("Upload: Datasets.zip")
uploaded = files.upload()

# Or from Google Drive:
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/AlgorimDatasets/Datasets.zip /content/

os.makedirs('/content/datasets', exist_ok=True)
if os.path.exists('/content/Datasets.zip'):
    with zipfile.ZipFile('/content/Datasets.zip') as zf:
        zf.extractall('/content/datasets')
    print("✅ Extracted Datasets.zip")

for f in glob.glob('/content/datasets/**/*.json', recursive=True):
    print(f"  {f}")


## 🧩 4 · Build Agentic Function-Calling Dataset

In [ ]:
import json, glob, random

SYSTEM_GEN = (
    "You are AlgorimGen, an agentic AI system specialized in Algorim (algo a47). "
    "You can execute programs, compile bytecode, debug errors, and generate visual code images. "
    "You reason step-by-step using <thinking> blocks and support tool calls:\n"
    "  /execute <code>   — runs the Algorim VM and returns output + trace\n"
    "  /compile <code>   — compiles to bytecode and returns compiled output\n"
    "  /debug <code>     — identifies syntax/logic errors and fixes them\n"
    "  /imagine <code>   — generates a visual PNG of the code\n"
    "Always show your reasoning inside <thinking>...</thinking> before calling a tool."
)

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except:
                    pass
    return records

# ── Load all dataset files ─────────────────────────────────────────
datasets_dir = '/content/datasets'
exec_traces = []
extra_data  = []
training    = []

for f in glob.glob(f'{datasets_dir}/**/*.json', recursive=True):
    fname = os.path.basename(f).lower()
    if 'execution' in fname or 'trace' in fname:
        exec_traces.extend(load_jsonl(f))
        print(f"Loaded execution traces: {f} ({len(exec_traces)} records)")
    elif 'extra' in fname:
        extra_data.extend(load_jsonl(f))
        print(f"Loaded extra data: {f} ({len(extra_data)} records)")
    elif 'training' in fname and 'vision' not in fname:
        training.extend(load_jsonl(f))
        print(f"Loaded training data: {f} ({len(training)} records)")

# ── Extract commands category from training ────────────────────────
commands_records = [r for r in training if r.get('metadata', {}).get('category') == 'commands']
print(f"\n  Commands-category records: {len(commands_records)}")
print(f"  Execution traces:          {len(exec_traces)}")
print(f"  Extra agentic:             {len(extra_data)}")

all_exec = exec_traces + extra_data
print(f"  Total execution samples:   {len(all_exec)}")


## 🔧 4b · Format Agentic Training Samples

In [ ]:
import os

def format_execution_sample(rec):
    """Format execution trace record as agentic tool-call conversation."""
    code   = rec.get('code', '')
    q      = rec.get('question', 'Run this Algorim program and show the output.')
    answer = rec.get('answer', code)
    trace  = rec.get('execution_trace', '')
    vm_out = rec.get('vm_output', [])
    vm_steps = rec.get('vm_steps', 0)
    vm_status = rec.get('vm_status', 'success')
    thinking = rec.get('thinking', '')
    compiled = rec.get('compiled_output', '')
    topic  = rec.get('topic', 'unknown')
    cmds   = rec.get('commands_used', [])

    # Build tool call response
    tool_call = "/execute\n```\n" + code.strip() + "\n```"
    tool_result = (
        f"VM Status: {vm_status}\n"
        f"Output: {vm_out}\n"
        f"Steps: {vm_steps}\n"
        f"Trace: {trace}"
    ) if trace else f"VM Status: {vm_status}\nOutput: {vm_out}\nSteps: {vm_steps}"

    if not thinking:
        thinking = (
            f"<thinking>\n"
            f"1. Parse the Algorim program for topic: {topic}\n"
            f"2. Identify entry point and control flow\n"
            f"3. Execute with /execute tool\n"
            f"4. Collect VM output and trace\n"
            f"</thinking>"
        )

    assistant_response = (
        f"{thinking}\n\n"
        f"{tool_call}\n\n"
        f"**VM Result:**\n```\n{tool_result}\n```\n\n"
    )
    if compiled:
        assistant_response += f"**Compiled:** {compiled}\n"
    assistant_response += f"\n**Output:** `{', '.join(str(o) for o in vm_out) if vm_out else 'no output'}`"

    return (
        f"<|im_start|>system\n{SYSTEM_GEN}<|im_end|>\n"
        f"<|im_start|>user\n{q}\n\n```algorim\n{code.strip()}\n```<|im_end|>\n"
        f"<|im_start|>assistant\n{assistant_response}<|im_end|>"
    )

def format_commands_sample(rec):
    """Format commands-category record preserving tool call patterns."""
    msgs = rec.get('messages', [])
    parts = []
    for m in msgs:
        role, content = m['role'], m['content']
        if role == 'system':
            # Replace system prompt with AlgorimGen one
            parts.append(f"<|im_start|>system\n{SYSTEM_GEN}<|im_end|>")
        elif role == 'user':
            parts.append(f"<|im_start|>user\n{content}<|im_end|>")
        elif role == 'assistant':
            parts.append(f"<|im_start|>assistant\n{content}<|im_end|>")
    return "\n".join(parts)

# ── Build all agentic samples ──────────────────────────────────────
all_samples = []

for rec in all_exec:
    txt = format_execution_sample(rec)
    all_samples.append({"text": txt})

for rec in commands_records:
    txt = format_commands_sample(rec)
    all_samples.append({"text": txt})

# Augment with ALL training records (AlgorimGen also writes/debugs code)
for rec in training:
    msgs = rec.get('messages', [])
    if not msgs:
        continue
    parts = []
    for m in msgs:
        role, content = m['role'], m['content']
        if role == 'system':
            parts.append(f"<|im_start|>system\n{SYSTEM_GEN}<|im_end|>")
        elif role == 'user':
            parts.append(f"<|im_start|>user\n{content}<|im_end|>")
        elif role == 'assistant':
            parts.append(f"<|im_start|>assistant\n{content}<|im_end|>")
    all_samples.append({"text": "\n".join(parts)})

random.shuffle(all_samples)
print(f"\n📊 Total AlgorimGen training samples: {len(all_samples)}")
print(f"   Execution traces: {len(all_exec)}")
print(f"   Commands records: {len(commands_records)}")
print(f"   Full training:    {len(training)}")
print("\nSample preview (execution format):")
print(all_samples[0]['text'][:600])


## 🤖 5 · Model Selection

In [ ]:
GEN_MODELS = {
    # DeepSeek-R1 (strongest reasoning + execution tracing)
    "DeepSeek-R1-7B":   "unsloth/DeepSeek-R1-Distill-Qwen-7B-unsloth-bnb-4bit",
    "DeepSeek-R1-1.5B": "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-bnb-4bit",
    # Qwen2.5 (strong function-calling capability)
    "Qwen2.5-7B":       "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    "Qwen2.5-3B":       "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    "Qwen2.5-0.5B":     "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    # Qwen2.5-Coder (best for code + execution)
    "Qwen2.5-Coder-7B": "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    "Qwen2.5-Coder-3B": "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
    "Qwen2.5-Coder-0.5B":"unsloth/Qwen2.5-Coder-0.5B-Instruct-bnb-4bit",
}

SELECTED    = "DeepSeek-R1-7B"   # ← change this
BASE_MODEL  = GEN_MODELS[SELECTED]
OUTPUT_NAME = "AlgorimGen"
MAX_SEQ_LEN = 2048
LORA_R      = 16
LORA_ALPHA  = 32

print(f"✅ Selected: {SELECTED} → {BASE_MODEL}")


## 🔧 6 · Load Model + LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = BASE_MODEL,
    max_seq_length= MAX_SEQ_LEN,
    dtype         = None,
    load_in_4bit  = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                       "gate_proj","up_proj","down_proj"],
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

model.print_trainable_parameters()


## 📋 7 · Prepare Dataset

In [ ]:
from datasets import Dataset

hf_dataset = Dataset.from_list(all_samples)
split      = hf_dataset.train_test_split(test_size=0.05, seed=42)
train_ds   = split['train']
eval_ds    = split['test']
print(f"Train: {len(train_ds):,}  |  Eval: {len(eval_ds):,}")


## 🏋️ 8 · Training

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps        = 10,
        num_train_epochs    = 3,
        learning_rate       = 2e-4,
        fp16  = not torch.cuda.is_bf16_supported(),
        bf16  =     torch.cuda.is_bf16_supported(),
        logging_steps       = 20,
        evaluation_strategy = "steps",
        eval_steps          = 100,
        save_strategy       = "steps",
        save_steps          = 200,
        output_dir          = f"/content/{OUTPUT_NAME}-checkpoints",
        optim               = "adamw_8bit",
        weight_decay        = 0.01,
        lr_scheduler_type   = "cosine",
        seed                = 42,
        report_to           = "none",
    ),
)

print("🚀 Starting AlgorimGen training...")
stats = trainer.train()
print(f"\n✅ Done! loss={stats.metrics['train_loss']:.4f}")


## 💾 9 · Save LoRA

In [ ]:
lora_path = f"/content/{OUTPUT_NAME}-lora"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"✅ LoRA → {lora_path}")


## 🔀 10 · Merge Full Model

In [ ]:
merged_path = f"/content/{OUTPUT_NAME}"
print("⏳ Merging...")
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
import os
total = sum(os.path.getsize(os.path.join(r,f)) for r,d,fs in os.walk(merged_path) for f in fs)
print(f"✅ Merged → {merged_path}  ({total/1e9:.2f} GB)")


## 📤 11 · GGUF F16

In [ ]:
gguf_path = f"/content/{OUTPUT_NAME}-GGUF"
print("⏳ GGUF F16...")
model.save_pretrained_gguf(gguf_path, tokenizer, quantization_method="f16")
import glob, os
for f in glob.glob(f"{gguf_path}/*.gguf"):
    print(f"  ✅ {f}  ({os.path.getsize(f)/1e9:.2f} GB)")


## 📤 12 · GGUF Q4_K_M

In [ ]:
print("⏳ GGUF Q4_K_M...")
model.save_pretrained_gguf(gguf_path, tokenizer, quantization_method="q4_k_m")
import glob, os
for f in glob.glob(f"{gguf_path}/*.gguf"):
    if 'Q4' in f.upper() or 'q4' in f:
        print(f"  ✅ {f}  ({os.path.getsize(f)/1e9:.2f} GB)")
print("\n🎉 AlgorimGen pipeline complete!")


## 🧪 13 · Agentic Inference Test

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

TEST = (
    "<|im_start|>system\n"
    + SYSTEM_GEN + "<|im_end|>\n"
    "<|im_start|>user\n"
    "Execute this Algorim program and show the output with trace:\n\n"
    "```algorim\n"
    "Action factorial(n: int): int\n"
    "begin\n"
    "    if (n <= 1) then\n"
    "        factorial <--- 1\n"
    "    else\n"
    "        factorial <--- n * factorial(n - 1)\n"
    "    endif\n"
    "end\n"
    "Action main()\n"
    "var\n"
    "    r: int\n"
    "begin\n"
    "    r <--- factorial(6)\n"
    "    print(r)\n"
    "end\n"
    "```<|im_end|>\n"
    "<|im_start|>assistant\n"
)

inputs = tokenizer(TEST, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens = 400,
    temperature    = 0.7,
    do_sample      = True,
    pad_token_id   = tokenizer.eos_token_id,
)
resp = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print("=" * 60)
print("AlgorimGen — Agentic Execution Response:")
print("=" * 60)
print(resp)
